# Kien Doan TTS — Minh Anh

> **GPU:** T4 16GB (tu dong) · **Thoi gian:** ~3 phut lan dau, ~30s/lan sau

## Huong dan

1. **Runtime** → **Run all** (Ctrl+F9)
2. Nhap van ban → Click **Tao giong noi**
3. Nghe + tai file WAV ve

> Powered by OmniVoice (MIT License) — github.com/k2-fsa/OmniVoice


In [ ]:
print('Dang cai dat (~2 phut)...')
!pip install -q omnivoice gradio numpy torch
print('Cai dat hoan tat!')

# Tai giong mau 10s tu voice-notebooks repo
!wget -q https://raw.githubusercontent.com/doanquangkien/voice-notebooks/main/samples/minh-anh.mp3 -O voice_sample.mp3
print('Voice sample loaded!')


In [ ]:
print('Dang khoi dong Omnivoice... (lan dau ~5 phut, lan sau ~30 giay)')

import logging, os, re, time
import numpy as np
import torch
import gradio as gr
from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Wait for GPU
for i in range(30):
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        break
    time.sleep(1)
else:
    print('GPU not available — make sure Runtime > Change runtime type > T4 GPU')

# Load model
DEVICE = get_best_device()
logger.info(f'Loading OmniVoice on {{DEVICE}}...')
model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True
)
SAMPLING_RATE = model.sampling_rate
logger.info(f'Model ready — SR: {{SAMPLING_RATE}}Hz')

# Voice clone prompt
logger.info('Creating VoiceClonePrompt...')
VOICE_PROMPT = model.create_voice_clone_prompt(ref_audio='voice_sample.mp3')
logger.info('Voice prompt ready — Minh Anh')
logger.info('Voice prompt ready — Minh Anh')

# Generate function
GEN_CFG = OmniVoiceGenerationConfig(
    num_step=32, guidance_scale=1.8,
    denoise=True, preprocess_prompt=True, postprocess_output=True,
    position_temperature=5.0, class_temperature=0.2,
    pad_duration=0.1, fade_duration=0.1,
)

def generate_voice(text: str):
    text = text.strip()
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    if not paragraphs:
        return None
    if len(paragraphs) == 1:
        audio = model.generate(
            text=paragraphs[0], voice_clone_prompt=VOICE_PROMPT,
            language='vi', speed=0.95, generation_config=GEN_CFG
        )[0]
    else:
        audios = []
        for i, p in enumerate(paragraphs):
            a = model.generate(
                text=p, voice_clone_prompt=VOICE_PROMPT,
                language='vi', speed=0.95, generation_config=GEN_CFG
            )[0]
            audios.append(a)
            if i < len(paragraphs) - 1:
                audios.append(np.zeros(int(SAMPLING_RATE * 0.3)))
        audio = np.concatenate(audios)
    waveform = (audio * 32767).astype(np.int16)
    return (SAMPLING_RATE, waveform)

# Launch UI
print('Khoi dong giao dien Kien Doan TTS — Minh Anh...')
CSS = ".gradio-container{{max-width:720px!important;margin:0 auto!important;padding:16px!important}}footer{{display:none!important}}"
THEME = gr.themes.Soft(primary_hue='indigo')
with gr.Blocks(title='Kien Doan TTS — Minh Anh', theme=THEME, css=CSS) as demo:
    gr.Markdown('# Kien Doan TTS\\nMinh Anh · Giong nu mien Bac, nhe nhang, truyen cam, phu hop marketing, quang cao, sach noi.')
    t = gr.Textbox(label='Nhap van ban', lines=4, placeholder='Nhap van ban ban muon chuyen thanh giong noi...')
    btn = gr.Button('Tao giong noi', variant='primary')
    out = gr.Audio(label='Ket qua')
    btn.click(generate_voice, inputs=[t], outputs=[out], concurrency_limit=1)

demo.launch(server_name='0.0.0.0', server_port=7860, share=True, theme=THEME, css=CSS)
